# Guanajuato SourceBuilder — reproducibility and QA

Reference coordinate: **20.4243195, -101.1466639** (not a verified parcel). This notebook does not modify existing run outputs; commands to fetch NASA/ISRIC data are opt-in and deliberately disabled by default.

Run from the repository root in the Python 3.11 environment with `source` dependencies installed.

In [ ]:
from pathlib import Path
import json
from gradientdescentfarmers.source_builder.adapter import REPO_ROOT, NASAProfileSourceBuilder, load_source_config
from gradientdescentfarmers.source_builder.core.audit import audit_profile

print('Repository:', REPO_ROOT)
assert (REPO_ROOT/'resources/guanajuato_reference.json').exists()
farmer = json.loads((REPO_ROOT/'resources/guanajuato_reference.json').read_text())
assert (farmer['latitude'], farmer['longitude']) == (20.4243195, -101.1466639)
print('Farm ID:', farmer['farm_id'])

## 1. Preview: schema and contract validation without remote requests

This exports an **unpopulated schema manifest**, not a validated climate dataset.

In [ ]:
cfg=load_source_config(REPO_ROOT/'resources/guanajuato_baseline.json')
cfg['run']=False
preview=NASAProfileSourceBuilder().build(cfg)
print(json.dumps({k:v for k,v in preview.items() if k not in ('tables','model_readiness')},indent=2))
assert preview['source_status']['nasa_power']=='not_requested'

## 2. Baseline retrieval (explicit opt-in)

Set `RUN_BASELINE=True` manually only when online. Expected providers: NASA POWER, SRTM, SoilGrids, MODIS (3 products), World Bank. Real performance and QA are **not** claimed before running. Each run is stored under `data/processed/guanajuato_valle_demo/runs/<run_id>`.


In [ ]:
RUN_BASELINE=False
baseline=None
if RUN_BASELINE:
    cfg=load_source_config(REPO_ROOT/'resources/guanajuato_baseline.json')
    cfg['run']=True
    baseline=NASAProfileSourceBuilder().build(cfg)
    print(baseline['run_id'],baseline['build_state'])
else:
    print('Skipped real network calls')

## 3. Baseline audit

Uses the saved manifest and SHA-256 file checks. This does not re-download or mutate scientific source tables.

In [ ]:
from gradientdescentfarmers.source_builder.core.audit import audit_sites
root=REPO_ROOT/'data/processed'
latest=root/farmer['farm_id']/'latest.json'
if latest.exists():
    report=audit_sites(root,sites=(farmer['farm_id'],))
    print(json.dumps(report,indent=2,default=str)[:5000])
else:
    print('No real baseline run yet; do not infer QA success from preview')

## 4. Earthdata smoke test (explicit opt-in)

Requires Earthdata credentials. Two GPM half-hours are **not a full day**. Small SMAP window. Review `availability.json`, table QA and extraction warnings. Original HDF5 cache should not be deleted.

In [ ]:
RUN_SMOKE=False
if RUN_SMOKE:
    cfg=load_source_config(REPO_ROOT/'resources/guanajuato_smoke.json')
    cfg['run']=True
    smoke=NASAProfileSourceBuilder().build(cfg)
    print(smoke['run_id'],smoke['source_status'].get('gpm_imerg_v07'),smoke['source_status'].get('smap_l4'))
else:
    print('Earthdata smoke skipped')

## 5. GPM 48-slot validation (gate)

Run the Full Day only after smoke and resumable download recovery are verified. For GPM, a valid day means 48 distinct UTC half-hours with valid precipitation observations, no duplicated timestamps and correct hourly-rate-to-30-min conversion.

In [ ]:
RUN_FULL_DAY=False
if RUN_FULL_DAY:
    raise RuntimeError('Full-day retrieval remains gated until robust downloader integration is tested')
print('Full-day retrieval gated: no accidental 48-file download')

## 6. Offline scientific regression

From terminal: `python -m pytest tests/source_builder tests/test_scaffold.py -q`. Compare QA fields and archived provenance; never claim observed yield/predicted profit from baseline weather/soil.